# MapBiomas burned area 2024: restoration and readiness footprint

**Business question.** Which provinces carry the largest 2024 burned-area footprint and therefore require the strongest recovery, prevention, and monitoring allocation? This notebook uses the MapBiomas annual burned-area raster only.

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'data').exists())
adm1 = gpd.read_file(ROOT / 'data/raw/boundaries/idn_adm1.geojson').to_crs('EPSG:4326')
name_col = next(c for c in ['shapeName', 'NAME_1', 'name'] if c in adm1.columns)
kalimantan = adm1[adm1[name_col].astype(str).str.contains('Kalimantan', case=False, na=False)].copy()
raster_path = ROOT / 'data/raw/mapbiomas/burned_area_2024_collection2.tif'
with rasterio.open(raster_path) as src:
    window = from_bounds(*kalimantan.total_bounds, transform=src.transform).round_offsets().round_lengths()
    scale = min(1.0, 2000 / max(window.width, window.height))
    height, width = max(1, round(window.height * scale)), max(1, round(window.width * scale))
    full = src.read(1, window=window, out_shape=(height, width), resampling=Resampling.average, masked=True)
    sample_transform = src.window_transform(window) * Affine.scale(window.width / width, window.height / height)
inside = geometry_mask([geometry.__geo_interface__ for geometry in kalimantan.geometry], out_shape=full.shape, transform=sample_transform, invert=True)
full = np.ma.array(full, mask=np.ma.getmaskarray(full) | ~inside)
rows = []
valid_cells = ~np.ma.getmaskarray(full)
for _, province in kalimantan.iterrows():
    province_inside = geometry_mask([province.geometry.__geo_interface__], out_shape=full.shape, transform=sample_transform, invert=True)
    values = np.asarray(full.data[province_inside & valid_cells])
    rows.append({'province': province[name_col], 'burned_pixel_share_pct': float(np.clip(values, 0, 1).mean() * 100), 'valid_blocks': int(len(values))})

province_burn = pd.DataFrame(rows).sort_values('burned_pixel_share_pct', ascending=False)
province_burn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
axes[0].barh(province_burn['province'][::-1], province_burn['burned_pixel_share_pct'][::-1], color='#b23a48')
axes[0].set(title='2024 burned-pixel share by province', xlabel='Share of valid raster pixels (%)')
axes[1].imshow(full, cmap='YlOrRd', interpolation='nearest')
axes[1].set(title='MapBiomas 2024 burned-area footprint', xticks=[], yticks=[])
plt.show()

leader = province_burn.iloc[0]
print(f"Business insight: {leader['province']} has the highest burned-pixel share in the 2024 raster ({leader['burned_pixel_share_pct']:.2f}%).")
print('Recommended action: prioritise this province for post-fire recovery checks, prevention-budget review, and dry-season readiness audits; use burned-pixel share as a comparable exposure indicator, not an exact hectare estimate.')

In [ ]:
# EDA tambahan: apakah footprint tersebar merata atau terkonsentrasi pada sedikit klaster?
intensity = np.sort(np.clip(full.compressed().astype(float), 0, 1))[-1::-1]
intensity = intensity[intensity > 0]
cumulative = np.cumsum(intensity) / intensity.sum()
block_share = np.arange(1, len(intensity) + 1) / len(intensity)
plt.figure(figsize=(8, 4))
plt.plot(block_share * 100, cumulative * 100, color='#b23a48', linewidth=2, label='Footprint terbakar kumulatif')
plt.plot([0, 100], [0, 100], '--', color='grey', label='Sebaran merata')
plt.title('Konsentrasi spasial footprint terbakar 2024'); plt.xlabel('Blok terbakar terurut dari terbesar (%)'); plt.ylabel('Footprint kumulatif (%)'); plt.grid(alpha=.25); plt.legend(); plt.show()
top_tenth = cumulative[int(np.ceil(len(intensity) * .10)) - 1] * 100
print(f'Business insight: 10% blok footprint terbesar menyumbang {top_tenth:.1f}% footprint yang terdeteksi. Gunakan klaster terbesar sebagai daftar audit pemulihan, akses, dan pencegahan tahun berikutnya; ini bukan indikator intensitas api.')

In [ ]:
# Perbandingan annual resmi 2023–2024 per provinsi.
annual = pd.read_parquet(ROOT / 'data/interim/mapbiomas_province_year_features.parquet')[['province', 'source_year', 'burned_share_pct']]
annual = annual[annual['source_year'].isin([2023, 2024])]
assert len(annual) == 10 and annual['province'].nunique() == 5
comparison = annual.pivot(index='province', columns='source_year', values='burned_share_pct').sort_values(2023, ascending=False)
ax = comparison.plot.barh(figsize=(10, 5), color={2023: '#d68c45', 2024: '#b23a48'}, title='Perbandingan footprint terbakar annual per provinsi')
ax.invert_yaxis(); ax.set(xlabel='Pangsa piksel valid terbakar (%)', ylabel=''); ax.grid(axis='x', alpha=.25)
plt.show()
change = comparison[2024] - comparison[2023]
print(f'Insight bisnis: penurunan terbesar terjadi di {change.idxmin()} ({change.min():.2f} pp). Verifikasi penyebabnya—cuaca, intervensi, kondisi lanskap, atau observabilitas—sebelum mengubah alokasi pencegahan.')
print('Produk monitoring bulanan 2026 tersedia untuk Januari–Juni saja. Jangan mengisi 2025 atau menjumlahkan footprint bulanan sebagai angka annual karena sel dapat muncul pada lebih dari satu bulan.')

## Decision notes

- Produk annual tersedia untuk 2023 dan 2024; monitoring 2026 yang tersedia baru Januari–Juni. Tidak ada nilai 2025 yang diproyeksikan.
- Compare shares rather than raw pixel counts because provincial extents differ.
- Validate the top areas in field planning before committing restoration budgets.